In [ ]:
# NOTEBOOK 32 - which regional stations around Delhi have usable history (for a wind-aware graph that reaches beyond the city)?
# Idea being checked: Delhi's winter air arrives from the north-west (Haryana, Punjab). A graph that also contains upwind stations
# 25-250 km away, linked with a travel-time lag, gives the model information a static 12 km Delhi graph cannot use.
# This notebook only COUNTS what exists: monitors with a PM2.5 sensor outside Delhi, in a box around the NCR / Haryana / Punjab corridor,
# split by distance from Delhi and by compass sector, and how much of their hourly data really exists in sampled months of 2024-2026.
# It reads the OpenAQ location list already cached by notebook 23 and about 60 small archive files per station (no key needed).
# Nothing is trained and no existing file is changed. About 5-10 minutes.
# HOW TO CHECK: this cell only imports and sets options.
import os, json, io, time, datetime as dt
import numpy as np, pandas as pd, requests
from concurrent.futures import ThreadPoolExecutor

V2_DIR = r'D:\FYP\Eco-Twin\v2_clean'
WORK = os.path.join(V2_DIR, 'crosscity_cache')
S3 = 'https://openaq-data-archive.s3.amazonaws.com/records/csv.gz'
DELHI = (28.64, 77.17)                                   # centre of the 37-station Delhi network
BOX = (27.2, 31.8, 73.6, 78.6)                           # south, north, west, east: NCR, Haryana, Punjab, west UP
SAMPLE_MONTHS = ['2024-01', '2024-06', '2024-11', '2025-03', '2025-11', '2026-02']      # 10 days from each month are read
DAYS = range(1, 11)
MIN_COVER = 0.70
print('box', BOX, '| sample months', SAMPLE_MONTHS)

In [ ]:
# 1) candidate monitors from the cached OpenAQ list: fixed reference-grade, PM2.5 sensor, inside the box, outside Delhi itself
locs = json.load(open(os.path.join(WORK, 'india_locations.json')))
st = pd.read_excel(os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')); st.columns = st.columns.str.strip().str.lower()
dlat, dlon = st['latitude'].values.astype(float), st['longitude'].values.astype(float)

def km(a, b, c, d):
    p1, p2 = np.radians(a), np.radians(c); x = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(d - b) / 2) ** 2
    return 12742 * np.arcsin(np.sqrt(x))
def bearing(lat1, lon1, lat2, lon2):                       # compass bearing from Delhi TO the station (0 = north, 270 = west)
    p1, p2, dl = np.radians(lat1), np.radians(lat2), np.radians(lon2 - lon1)
    return (np.degrees(np.arctan2(np.sin(dl) * np.cos(p2), np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl))) + 360) % 360

rows = []
for L in locs:
    c = L.get('coordinates') or {}; lat, lon = c.get('latitude'), c.get('longitude')
    if lat is None or L.get('isMobile') or not L.get('isMonitor'): continue
    if not (BOX[0] <= lat <= BOX[1] and BOX[2] <= lon <= BOX[3]): continue
    if not any((s.get('parameter') or {}).get('name') == 'pm25' for s in L.get('sensors', [])): continue
    if km(dlat, dlon, lat, lon).min() <= 1.5: continue                                  # one of our 37 Delhi stations
    dist = float(km(DELHI[0], DELHI[1], lat, lon))
    rows.append({'location_id': L['id'], 'name': L['name'], 'lat': lat, 'lon': lon, 'km_from_delhi': dist,
                 'bearing_deg': float(bearing(DELHI[0], DELHI[1], lat, lon)), 'provider': (L.get('provider') or {}).get('name'),
                 'first': (L.get('datetimeFirst') or {}).get('utc'), 'last': (L.get('datetimeLast') or {}).get('utc')})
cand = pd.DataFrame(rows)
cand['first'] = pd.to_datetime(cand['first'], utc=True, errors='coerce'); cand['last'] = pd.to_datetime(cand['last'], utc=True, errors='coerce')
print('monitors with a PM2.5 sensor in the box (outside Delhi):', len(cand))
# these two filters only skip stations that cannot have the history needed
long_enough = cand[(cand['first'] <= pd.Timestamp('2024-02-15', tz='UTC')) & (cand['last'] >= pd.Timestamp('2026-03-01', tz='UTC'))].reset_index(drop=True)
print('with records starting by Feb 2024 and still reporting in Mar 2026:', len(long_enough))
cand.to_csv(os.path.join(WORK, 'regional_candidates_all.csv'), index=False)

# HOW TO CHECK: a few hundred monitors in the box (many are low-cost sensors excluded by the monitor filter); far fewer pass the history filter.

In [ ]:
# 2) coverage in the sampled months, from the public archive
def read_day(url):
    try:
        r = requests.get(url, timeout=40)
        if r.status_code != 200: return None
        d = pd.read_csv(io.BytesIO(r.content), compression='gzip', usecols=['datetime', 'parameter', 'value'])
        d = d[(d['parameter'] == 'pm25') & (d['value'] >= 0) & (d['value'] < 1000)]
        return pd.to_datetime(d['datetime'], utc=True).dt.tz_convert('Asia/Kolkata').dt.floor('h').nunique() if len(d) else 0
    except Exception:
        return None

def station_cover(loc_id):
    urls, per = [], {}
    for m in SAMPLE_MONTHS:
        y, mo = int(m[:4]), int(m[5:])
        urls += [(m, f'{S3}/locationid={loc_id}/year={y}/month={mo:02d}/location-{loc_id}-{y}{mo:02d}{d:02d}.csv.gz') for d in DAYS]
    with ThreadPoolExecutor(max_workers=8) as ex:
        res = list(ex.map(lambda t: read_day(t[1]), urls))
    for (m, _), h in zip(urls, res):
        per.setdefault(m, []).append(0 if h is None else h)
    return {m: float(np.sum(v) / (24 * len(DAYS))) for m, v in per.items()}

cache = os.path.join(WORK, 'regional_cover.json')
cov = json.load(open(cache)) if os.path.exists(cache) else {}
todo = [int(i) for i in long_enough['location_id'] if str(int(i)) not in cov]
print('stations to measure:', len(todo)); t0 = time.time()
for k, lid in enumerate(todo):
    cov[str(lid)] = station_cover(lid)
    if (k + 1) % 10 == 0: print(f'  {k + 1}/{len(todo)}  {time.time() - t0:.0f}s'); json.dump(cov, open(cache, 'w'))
json.dump(cov, open(cache, 'w'))
print(f'done in {time.time() - t0:.0f}s')

In [ ]:
# 3) results: usable stations by distance ring and compass sector
d = long_enough.copy()
for m in SAMPLE_MONTHS: d[m] = d['location_id'].map(lambda i: cov.get(str(int(i)), {}).get(m, 0.0))
d['mean_cover'] = d[SAMPLE_MONTHS].mean(axis=1); d['months_ok'] = (d[SAMPLE_MONTHS] >= MIN_COVER).sum(axis=1)
d['usable'] = (d['mean_cover'] >= MIN_COVER) & (d['months_ok'] >= len(SAMPLE_MONTHS) - 1)     # at least 5 of 6 sampled months are well covered
d['ring'] = pd.cut(d['km_from_delhi'], [0, 25, 60, 120, 250, 1000], labels=['0-25 km', '25-60 km', '60-120 km', '120-250 km', '250+ km'])
d['sector'] = pd.cut((d['bearing_deg'] + 45) % 360, [0, 90, 180, 270, 360], labels=['N', 'E', 'S', 'W'], right=False)     # N = 315-45, W = 225-315
d['NW'] = (d['bearing_deg'] >= 270) | (d['bearing_deg'] <= 25)                                  # the winter transport side: west through north-north-east
d.sort_values('km_from_delhi').to_csv(os.path.join(WORK, 'regional_stations.csv'), index=False)

print('usable = PM2.5 mostly present (>= 70 %) in at least 5 of the 6 sampled months of 2024-2026')
print()
print('usable stations by distance from Delhi and side:')
tab = d[d['usable']].pivot_table(index='ring', columns=pd.cut(d[d['usable']]['bearing_deg'], [-1, 90, 180, 270, 360], labels=['N-E (0-90)', 'E-S (90-180)', 'S-W (180-270)', 'W-N (270-360)']),
                                 values='location_id', aggfunc='count', observed=False).fillna(0).astype(int)
print(tab.to_string())
print()
print('total usable:', int(d['usable'].sum()), '| usable on the north-west side (270-25 deg):', int((d['usable'] & d['NW']).sum()))
print()
print('usable stations on the north-west side, nearest first:')
x = d[d['usable'] & d['NW']].sort_values('km_from_delhi')
print(x[['name', 'km_from_delhi', 'bearing_deg', 'mean_cover', 'months_ok']].round(2).head(40).to_string(index=False))

# HOW TO READ: for a transport graph we want several usable stations on the north-west side at 25-250 km, ideally 8 or more.
# A handful (under 4) means there is not enough upwind information to build the extended graph.